# Piece-Square Training Notebook

End-to-end pipeline:
1. Load feature shards generated by `analysis/game_feature_extract.ipynb` (via helper cells below).
2. Fit a regularized linear model to predict game outcomes from piece-square occupancy.
3. Evaluate on a validation split.
4. Export learned modifiers to `jsons/location_modifiers.json` and (optionally) rewrite `Chess/LocationModifer.pm`.

## Prerequisites
- Create `.npz` shards by enabling `BUILD_SHARDS_FROM_PGN` below; the notebook executes `analysis/game_feature_extract.ipynb` directly.
- Install dependencies:
  ```bash
  python -m pip install numpy pandas scikit-learn matplotlib
  ```

In [ ]:
import io
import json
import math
import os
import subprocess
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    r2_score,
)

In [ ]:
def _env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def _resolve_path(raw: str | None, default_path: Path) -> Path:
    if raw is None or not raw.strip():
        return default_path
    parsed = Path(raw.strip())
    if parsed.is_absolute():
        return parsed
    return (REPO_ROOT / parsed).resolve()


REPO_ROOT = Path.cwd().resolve()
SHARD_DIR = _resolve_path(os.getenv("LOCATION_TRAINING_SHARD_DIR"), REPO_ROOT / "data" / "processed")
PGN_PATH = _resolve_path(os.getenv("LOCATION_TRAINING_PGN_PATH"), REPO_ROOT / "data" / "lichess_games_export.pgn")
OUTPUT_JSON = _resolve_path(os.getenv("LOCATION_TRAINING_OUTPUT_JSON"), REPO_ROOT / "data" / "location_modifiers.json")
UPDATE_SCRIPT = _resolve_path(os.getenv("LOCATION_TRAINING_UPDATE_SCRIPT"), REPO_ROOT / "scripts" / "update_location_modifiers.pl")
FEATURE_EXTRACT_NOTEBOOK = _resolve_path(os.getenv("LOCATION_TRAINING_FEATURE_EXTRACT_NOTEBOOK"), REPO_ROOT / "analysis" / "game_feature_extract.ipynb")

# Input source mode:
# - "existing_shards": use SHARD_DIR as-is
# - "existing_pgn": use PGN_PATH then (optionally) rebuild shards
# - "game_url_log": consume data/lichess_game_urls.log into PGN_PATH then rebuild shards
# - "lichess_db_urls": download one or more Lichess DB URLs into PGN_PATH then rebuild shards
DATA_SOURCE_MODE = os.getenv("LOCATION_TRAINING_DATA_SOURCE_MODE", "existing_shards").strip().lower()
GAME_URL_LOG_PATH = _resolve_path(os.getenv("LOCATION_TRAINING_GAME_URL_LOG_PATH"), REPO_ROOT / "data" / "lichess_game_urls.log")
DATA_INGRESS_SCRIPT_PATH = _resolve_path(os.getenv("LOCATION_TRAINING_DATA_INGRESS_SCRIPT_PATH"), REPO_ROOT / "scripts" / "data_ingress.sh")
DO_DATA_SCIENCE = _env_bool("DO_DATA_SCIENCE", False)
CLEAR_GAME_URL_LOG_AFTER_CONSUME = _env_bool("LOCATION_TRAINING_CLEAR_GAME_URL_LOG", DO_DATA_SCIENCE)

LICHESS_DB_URLS: List[str] = []
LICHESS_DB_TMP_DIR = _resolve_path(os.getenv("LOCATION_TRAINING_LICHESS_DB_TMP_DIR"), REPO_ROOT / "data" / "tmp")
KEEP_DB_DOWNLOADS = _env_bool("LOCATION_TRAINING_KEEP_DB_DOWNLOADS", False)

BUILD_SHARDS_FROM_PGN = _env_bool("LOCATION_TRAINING_BUILD_SHARDS_FROM_PGN", True)
CLEAR_SHARDS_BEFORE_BUILD = _env_bool("LOCATION_TRAINING_CLEAR_SHARDS_BEFORE_BUILD", True)
EXTRACT_SHARD_SIZE = int(os.getenv("LOCATION_TRAINING_EXTRACT_SHARD_SIZE", "25000"))
EXTRACT_MIN_ELO = int(os.getenv("LOCATION_TRAINING_EXTRACT_MIN_ELO", "1600"))
EXTRACT_MAX_GAMES = None
MAX_SHARDS_TO_LOAD = None

PIECES = [
    "PAWN",
    "KNIGHT",
    "BISHOP",
    "ROOK",
    "QUEEN",
    "KING",
    "OPP_PAWN",
    "OPP_KNIGHT",
    "OPP_BISHOP",
    "OPP_ROOK",
    "OPP_QUEEN",
    "OPP_KING",
]
SQUARES = [f"{file}{rank}" for file in "abcdefgh" for rank in range(1, 9)]
PHASES = ["opening", "middlegame", "endgame"]
FEATURES_PER_PIECE = 64
PHASE_FEATURES = len(PHASES)
TOTAL_PIECE_FEATURES = len(PIECES) * FEATURES_PER_PIECE
FEATURE_DIM = TOTAL_PIECE_FEATURES + PHASE_FEATURES
print(f"Expecting feature dimension {FEATURE_DIM}")
print(f"Data source mode: {DATA_SOURCE_MODE}")
print(f"Clear URL log after consume: {CLEAR_GAME_URL_LOG_AFTER_CONSUME}")

In [ ]:
def _nonempty_log_entries(path: Path) -> List[str]:
    if not path.exists():
        return []
    entries: List[str] = []
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        entries.append(line)
    return entries


def _truncate_file(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("", encoding="utf-8")


def _append_plain_pgn(src: Path, out_handle) -> None:
    with src.open("r", encoding="utf-8", errors="ignore") as handle:
        while True:
            chunk = handle.read(1 << 20)
            if not chunk:
                break
            out_handle.write(chunk)


def _append_zst_pgn(src: Path, out_handle) -> None:
    try:
        import zstandard as zstd
    except ImportError as exc:
        raise RuntimeError("zstandard package is required to decode .zst files") from exc

    with src.open("rb") as compressed:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(compressed) as reader:
            text_reader = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
            while True:
                chunk = text_reader.read(1 << 20)
                if not chunk:
                    break
                out_handle.write(chunk)


def _download_lichess_db_urls(urls: List[str], tmp_dir: Path, merged_pgn: Path, keep_downloads: bool) -> Path:
    if not urls:
        raise ValueError("LICHESS_DB_URLS is empty")

    tmp_dir.mkdir(parents=True, exist_ok=True)
    merged_pgn.parent.mkdir(parents=True, exist_ok=True)

    with merged_pgn.open("w", encoding="utf-8") as out:
        for idx, url in enumerate(urls, start=1):
            base_name = Path(url.split("?")[0]).name or f"lichess_{idx}.pgn"
            dest = tmp_dir / base_name
            cmd = ["curl", "-fL", "--retry", "3", "--retry-delay", "1", url, "-o", str(dest)]
            print("Downloading:", url)
            result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
            if result.returncode != 0:
                raise RuntimeError(f"curl failed for {url}: {result.stderr.strip() or result.stdout.strip()}")

            if dest.suffix.lower() == ".zst":
                _append_zst_pgn(dest, out)
            else:
                _append_plain_pgn(dest, out)
            out.write("\n\n")

            if not keep_downloads:
                dest.unlink(missing_ok=True)

    return merged_pgn


def _consume_game_url_log(log_path: Path, out_pgn: Path) -> Path:
    entries = _nonempty_log_entries(log_path)
    if not entries:
        raise ValueError(f"No URLs/game IDs found in {log_path}")
    if not DATA_INGRESS_SCRIPT_PATH.exists():
        raise FileNotFoundError(f"Missing data ingress script: {DATA_INGRESS_SCRIPT_PATH}")

    cmd = [
        str(DATA_INGRESS_SCRIPT_PATH),
        "OWN-URLS",
        "--own-url-log",
        str(log_path),
        "--own-pgn-output",
        str(out_pgn),
    ]
    if CLEAR_GAME_URL_LOG_AFTER_CONSUME:
        cmd.append("--clear-own-url-log")
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())

    if CLEAR_GAME_URL_LOG_AFTER_CONSUME:
        print(f"Cleared consumed URL log: {log_path}")

    return out_pgn


def _clear_existing_shards(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    for pattern in ("shard_*.npz", "shard_*.metadata.jsonl"):
        for file in output_dir.glob(pattern):
            file.unlink(missing_ok=True)


_feature_extract_ns: dict | None = None


def _load_feature_extractor_notebook_namespace(notebook_path: Path) -> dict:
    if not notebook_path.exists():
        raise FileNotFoundError(f"Missing extractor notebook: {notebook_path}")

    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    namespace: dict = {"__name__": "__notebook_loader__", "__file__": str(notebook_path)}

    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        if not source.strip():
            continue
        exec(compile(source, str(notebook_path), "exec"), namespace)

    if "main" not in namespace:
        raise RuntimeError(f"Extractor notebook did not define main(): {notebook_path}")
    return namespace


def _build_shards_from_pgn(pgn_path: Path, output_dir: Path) -> None:
    import sys

    if not pgn_path.exists():
        raise FileNotFoundError(f"Missing PGN file: {pgn_path}")

    if CLEAR_SHARDS_BEFORE_BUILD:
        _clear_existing_shards(output_dir)

    global _feature_extract_ns
    if _feature_extract_ns is None:
        _feature_extract_ns = _load_feature_extractor_notebook_namespace(FEATURE_EXTRACT_NOTEBOOK)

    cli_args = [
        str(pgn_path),
        "--output-dir",
        str(output_dir),
        "--shard-size",
        str(EXTRACT_SHARD_SIZE),
        "--min-elo",
        str(EXTRACT_MIN_ELO),
    ]
    if EXTRACT_MAX_GAMES is not None:
        cli_args.extend(["--max-games", str(int(EXTRACT_MAX_GAMES))])

    old_argv = list(sys.argv)
    try:
        sys.argv = [FEATURE_EXTRACT_NOTEBOOK.name] + cli_args
        _feature_extract_ns["main"]()
    finally:
        sys.argv = old_argv



def prepare_training_inputs() -> Tuple[Path | None, Path]:
    mode = DATA_SOURCE_MODE.strip().lower()

    if mode == "existing_shards":
        return None, SHARD_DIR

    if mode == "existing_pgn":
        if not PGN_PATH.exists():
            raise FileNotFoundError(f"Missing PGN file: {PGN_PATH}")
        if BUILD_SHARDS_FROM_PGN:
            _build_shards_from_pgn(PGN_PATH, SHARD_DIR)
        return PGN_PATH, SHARD_DIR

    if mode == "game_url_log":
        active_pgn = _consume_game_url_log(GAME_URL_LOG_PATH, PGN_PATH)
        if BUILD_SHARDS_FROM_PGN:
            _build_shards_from_pgn(active_pgn, SHARD_DIR)
        return active_pgn, SHARD_DIR

    if mode == "lichess_db_urls":
        active_pgn = _download_lichess_db_urls(LICHESS_DB_URLS, LICHESS_DB_TMP_DIR, PGN_PATH, KEEP_DB_DOWNLOADS)
        if BUILD_SHARDS_FROM_PGN:
            _build_shards_from_pgn(active_pgn, SHARD_DIR)
        return active_pgn, SHARD_DIR

    raise ValueError(f"Unsupported DATA_SOURCE_MODE: {DATA_SOURCE_MODE}")


def load_shards(shard_dir: Path, max_shards: int | None = None):
    arrays = []
    labels = []
    weights = []
    shard_paths = sorted(shard_dir.glob("shard_*.npz"))
    if max_shards:
        shard_paths = shard_paths[:max_shards]
    if not shard_paths:
        raise FileNotFoundError(f"No shards found in {shard_dir}")

    for path in shard_paths:
        data = np.load(path)
        arrays.append(data["features"])  # shape: (N, 12*64 + 3)
        labels.append(data["labels"])
        weights.append(data["weights"])
        print(f"Loaded {path.name}: {data['features'].shape[0]} samples")

    X = np.concatenate(arrays, axis=0)
    y = np.concatenate(labels, axis=0)
    w = np.concatenate(weights, axis=0)
    assert X.shape[1] == FEATURE_DIM
    print(f"Total samples: {X.shape[0]}")
    return X, y, w

In [ ]:
ACTIVE_PGN_PATH, ACTIVE_SHARD_DIR = prepare_training_inputs()
print(f"Active PGN: {ACTIVE_PGN_PATH}")
print(f"Active shard dir: {ACTIVE_SHARD_DIR}")

# Load a manageable subset for experimentation by setting max_shards.
X, y, sample_weights = load_shards(ACTIVE_SHARD_DIR, max_shards=MAX_SHARDS_TO_LOAD)

# Deterministic train/validation split (80/20) based on index.
indices = np.arange(X.shape[0])
split = int(0.8 * len(indices))
train_X, val_X = X[:split], X[split:]
train_y, val_y = y[:split], y[split:]
train_w, val_w = sample_weights[:split], sample_weights[split:]
print(f"Train: {train_X.shape[0]} rows, Val: {val_X.shape[0]} rows")

In [ ]:
alpha = 5.0  # tweak for regularization strength
model = Ridge(alpha=alpha, fit_intercept=False)
model.fit(train_X, train_y, sample_weight=train_w)
train_pred = model.predict(train_X)
val_pred = model.predict(val_X)
train_r2 = r2_score(train_y, train_pred, sample_weight=train_w)
val_r2 = r2_score(val_y, val_pred, sample_weight=val_w)
print(f"Train R^2: {train_r2:.4f} | Val R^2: {val_r2:.4f}")

In [ ]:
# Classification-oriented metrics derived from regression outputs

def discretize_outcomes(values, threshold=0.333):
    classes = np.zeros_like(values)
    classes[:] = 0
    classes[values > threshold] = 1
    classes[values < -threshold] = -1
    return classes

train_y_classes = discretize_outcomes(train_y)
val_y_classes = discretize_outcomes(val_y)
train_pred_classes = discretize_outcomes(train_pred)
val_pred_classes = discretize_outcomes(val_pred)

labels = [1, 0, -1]
label_names = {1: "Win", 0: "Draw", -1: "Loss"}

accuracy = accuracy_score(val_y_classes, val_pred_classes)
precision, recall, f1, support = precision_recall_fscore_support(
    val_y_classes, val_pred_classes, labels=labels, zero_division=0
)

metrics_table = [
    {
        "Class": label_names[label],
        "Precision": round(float(p), 3),
        "Recall": round(float(r), 3),
        "F1": round(float(f), 3),
        "Support": int(s),
    }
    for label, p, r, f, s in zip(labels, precision, recall, f1, support)
]

metrics_table.append(
    {
        "Class": "Macro Avg",
        "Precision": round(float(precision.mean()), 3),
        "Recall": round(float(recall.mean()), 3),
        "F1": round(float(f1.mean()), 3),
        "Support": int(support.sum()),
    }
)
metrics_table.append(
    {
        "Class": "Accuracy",
        "Precision": "-",
        "Recall": "-",
        "F1": round(float(accuracy), 3),
        "Support": int(support.sum()),
    }
)

metrics_df = pd.DataFrame(metrics_table)
display(metrics_df)
print(f"Validation accuracy: {accuracy:.3f}")


In [ ]:
# Confusion matrix visualization for validation set predictions
cm = confusion_matrix(val_y_classes, val_pred_classes, labels=labels)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([label_names[label] for label in labels])
ax.set_yticklabels([label_names[label] for label in labels])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Validation Confusion Matrix")
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, int(value), ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
coeffs = model.coef_[:TOTAL_PIECE_FEATURES]
print(f"Coefficient vector shape: {coeffs.shape}")

# Reshape into (piece, square)
modifier_matrix = coeffs.reshape(len(PIECES), FEATURES_PER_PIECE)

def normalize_piece_row(values: np.ndarray) -> np.ndarray:
    centered = values - values.mean()
    clipped = np.clip(centered, -40.0, 40.0)
    return clipped

json_payload: dict[str, dict[str, float]] = {}
for piece_idx, piece in enumerate(PIECES):
        normalized = normalize_piece_row(modifier_matrix[piece_idx])
        json_payload[piece] = {
            square: round(float(value), 3)
            for square, value in zip(SQUARES, normalized)
        }

print(f"Prepared modifiers for {len(json_payload)} piece types")

In [ ]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open("w", encoding="utf-8") as handle:
    json.dump(json_payload, handle, indent=2, sort_keys=True)
print(f"Wrote modifiers to {OUTPUT_JSON}")

In [ ]:
# Optional: regenerate Chess/LocationModifer.pm directly from the notebook.
run_perl_update = False  # set True to apply changes
if run_perl_update:
    result = subprocess.run(
        ["perl", str(UPDATE_SCRIPT), str(OUTPUT_JSON)],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        print(result.stdout)
    else:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Failed to update LocationModifer.pm")

## Next Steps
- Re-run `perl tests/perft.pl 4` and a self-play suite to validate the new tables.
- Commit `jsons/location_modifiers.json` and the regenerated `Chess/LocationModifer.pm` for review.